In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

# Start

## Data preparation

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        df = df[df['Non_Standard_Braking'] == 0]
        df = df[df['BC_BadStart'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'Dati(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source
        df['Malfunction'] = 0

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_Dati01.csv',
    'TestBrakefinal_data_Dati06.csv',
    'TestBrakefinal_data_Dati27.csv',
    'TestBrakefinal_data_Dati05.csv',
    'TestBrakefinal_data_Dati10.csv',
    'TestBrakefinal_data_Dati11.csv',
    'TestBrakefinal_data_Dati18.csv',
    'TestBrakefinal_data_Dati24.csv'
]

df = load_data(model_path, monorail_paths)
df['Malfunction'] = df['Malfunction'].astype(str)
print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

In [ ]:
df.head()

## PREPROCESS DATA FOR ML

In [ ]:
# Select the Loading Condition that is within 2 and 3 bar
# Instead of directly filtering, lets create WV_bin column for <2 bar, 2 to 3 bar, and >3 bar 
# based on WV_MeanPressure

from sklearn.model_selection import train_test_split

def preprocess_data(df, features, test_size,
                    label_col="label",
                    mal_col="Malfunction",
                    random_state=42):
    """
    Preprocess data:
      - filter WV_bin == 1
      - select features
      - split into train/test
      - keep both:
          * binary target (label_col)
          * malfunction codes (mal_col)
      - extract healthy-only training samples

    Parameters
    ----------
    df : pandas.DataFrame
        Full dataset.
    features : list of str
        Feature column names to use.
    test_size : float
        Proportion for test split.
    label_col : str, default 'label'
        Column name of the binary target (0/1).
    mal_col : str, default 'Malfunction'
        Column name of the malfunction code.

    Returns
    -------
    X_train, X_test : pandas.DataFrame
    y_train, y_test : pandas.Series
        Binary label (0/1).
    mal_train, mal_test : pandas.Series
        Malfunction codes (A,B,C,...,0) aligned to X_train/X_test.
    X_train_healthy : pandas.DataFrame
        Subset of X_train where y_train == 0.
    """

    df = df.copy()

    # Filter by WV_bin == 1 (pressure between 2 and 3 bar)
    df_filt = df[df["WV_bin"] == 1].copy()

    # Validate feature selection
    missing = [f for f in features if f not in df_filt.columns]
    if missing:
        raise ValueError(f"The following features are not in the dataframe: {missing}")

    # Features and targets
    X      = df_filt[features]
    y      = df_filt[label_col]       # main binary target
    y_mal  = df_filt[mal_col]         # malfunction codes

    # Joint split so everything stays aligned
    X_train, X_test, y_train, y_test, mal_train, mal_test = train_test_split(
        X, y, y_mal,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # Healthy training samples
    X_train_healthy = X_train[y_train == 0]

    print(f"Total samples: {len(df)}")
    print(f"Filtered samples (WV_bin==1): {len(df_filt)}")
    print(f"Training samples: {len(X_train)} (Healthy: {sum(y_train==0)}, Leakage: {sum(y_train==1)})")
    print(f"Training samples (healthy only): {len(X_train_healthy)}")
    print(f"Test samples: {len(X_test)} (Healthy: {sum(y_test==0)}, Leakage: {sum(y_test==1)})")

    return (
        X_train, X_test,
        y_train, y_test,
        mal_train, mal_test,
        X_train_healthy
    )


## Data Exploration Continues

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

# ------------------------------------------------------------
# 1. Data prep
# ------------------------------------------------------------
rng = np.random.default_rng(42)

df_plot = df.copy()
df_plot["y_jitter"] = rng.normal(0, 0.02, size=len(df_plot))

feat = "Total_power_efficiency"
df_plot[feat] = pd.to_numeric(df_plot[feat], errors="coerce")

df_filtered = df_plot[df_plot["WV_bin"] == 1].copy()
df_filtered = df_filtered.dropna(subset=[feat])

# ------------------------------------------------------------
# 2. DISTINCT color mapping (robust)
# ------------------------------------------------------------

# ---- Kit Source colors ----
sources = sorted(df_filtered["Source"].unique())
palette_source = sns.color_palette("tab10", n_colors=len(sources))
source_color_map = dict(zip(sources, palette_source))

df_filtered["color_source"] = df_filtered["Source"].map(source_color_map)

# ---- Leakage label colors (explicit & fixed) ----
label_color_map = {
    0: "#1f77b4",  # blue   (healthy)
    1: "#ff7f0e",  # orange (leakage)
    2: "#d62728",  # red    (severe)
}
df_filtered["color_label"] = df_filtered["label"].map(label_color_map)

# ------------------------------------------------------------
# 3. Plot
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# ---------------- Left: by Source ----------------
axes[0].scatter(
    df_filtered[feat],
    df_filtered["y_jitter"],
    c=df_filtered["color_source"],
    alpha=0.75,
    edgecolors="black",
    linewidths=0.3
)

axes[0].set_title("Total Power Efficiency distribution by Kit Source")
axes[0].set_xlabel(feat)
axes[0].set_yticks([])

# Correct legend (Source)
for s in sources:
    axes[0].scatter(
        [], [], 
        color=source_color_map[s],
        label=f"Source {s}",
        edgecolors="black"
    )
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: by Label ----------------
axes[1].scatter(
    df_filtered[feat],
    df_filtered["y_jitter"],
    c=df_filtered["color_label"],
    alpha=0.75,
    edgecolors="black",
    linewidths=0.3
)

axes[1].set_title("Total Power Efficiency distribution by Label")
axes[1].set_xlabel(feat)
axes[1].set_yticks([])

# Correct legend (Label)
for lab, col in label_color_map.items():
    axes[1].scatter([], [], color=col, label=f"Label {lab}", edgecolors="black")
axes[1].legend(title="Leakage Label", loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================
selected_features = ["Total_power_delay", "Total_power_efficiency",
                     "Total_energy_efficiency", "Std_delay_exp", "Total_energy_delay"]

rng = np.random.default_rng(42)   # reproducibility
df_plot = df.copy()
df_plot["y_jitter"] = rng.normal(0, 0.02, size=len(df_plot))

# Filter once (edit as you like)
df_filtered = df_plot[df_plot["WV_bin"] == 1].copy()
df_filtered = df_filtered[df_filtered["Total_energy_efficiency"] < 50].copy()
df_filtered = df_filtered[df_filtered["Total_power_efficiency"] < 10].copy()
# Palettes
palette_source = sns.color_palette("colorblind", n_colors=df_filtered["Source"].nunique())
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # orange
}

# Precompute stable category mappings (important for legends)
src_cat = df_filtered["Source"].astype("category")
src_codes = src_cat.cat.codes
src_levels = list(src_cat.cat.categories)
src_color_map = {level: palette_source[i] for i, level in enumerate(src_levels)}

lab_cat = df_filtered["label"].astype("category")
lab_codes = lab_cat.cat.codes
lab_levels = list(lab_cat.cat.categories)

# If your labels are literally {0,1}, the palette dict above is fine.
# If labels are strings like {"healthy","leakage"}, build a palette automatically:
if not set(lab_levels).issubset(set(label_palette.keys())):
    auto_pal = sns.color_palette("Set1", n_colors=len(lab_levels))
    label_color_map = {level: auto_pal[i] for i, level in enumerate(lab_levels)}
else:
    label_color_map = {level: label_palette[level] for level in lab_levels}

# ============================================================
# LOOP: one figure per feature
# ============================================================
for feat in selected_features:
    if feat not in df_filtered.columns:
        print(f"[skip] '{feat}' not found in df columns.")
        continue

    # numeric-safe (optional but recommended)
    x = pd.to_numeric(df_filtered[feat], errors="coerce")
    mask = x.notna()
    d = df_filtered.loc[mask].copy()
    x = x.loc[mask]

    # Colors
    colors_source = [src_color_map[s] for s in d["Source"]]
    colors_label  = [label_color_map[l] for l in d["label"]]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

    # --- Left: by Source ---
    axes[0].scatter(x, d["y_jitter"], c=colors_source, alpha=0.7)
    axes[0].set_title(f"{feat} distribution by Kit Source")
    axes[0].set_xlabel(feat)
    axes[0].set_yticks([])
    
    # Legend (Source)
    for s in src_levels:
        if (d["Source"] == s).any():
            axes[0].scatter([], [], c=[src_color_map[s]], label=str(s))
    axes[0].legend(title="Kit Source", loc="upper right")

    # --- Right: by label ---
    axes[1].scatter(x, d["y_jitter"], c=colors_label, alpha=0.7)
    axes[1].set_title(f"{feat} distribution by Label")
    axes[1].set_xlabel(feat)
    axes[1].set_yticks([])

    # Legend (Label)
    for lab in lab_levels:
        if (d["label"] == lab).any():
            axes[1].scatter([], [], c=[label_color_map[lab]], label=str(lab))
    axes[1].legend(title="Leakage Label", loc="upper right")

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================
selected_features = [
    "Total_power_delay",
    "Total_power_efficiency",
    "Total_energy_efficiency",
    "Std_delay_exp",
    "Total_energy_delay"
]

df_plot = df.copy()
df_plot = df_plot[df_plot["WV_bin"] == 1].copy()   # optional filter

# Make labels readable
df_plot["LeakageLabel"] = df_plot["label"].map({
    0: "Healthy",
    1: "Leakage"
})

# Consistent color semantics
palette_label = {
    "Healthy": "#1f77b4",   # blue
    "Leakage": "#ff7f0e",   # orange
}

# ============================================================
# LOOP: ONE FIGURE PER FEATURE
# ============================================================
for feat in selected_features:

    if feat not in df_plot.columns:
        print(f"[skip] {feat} not found")
        continue

    plt.figure(figsize=(14, 5))

    sns.boxplot(
        data=df_plot,
        x="Source",
        y=feat,
        hue="LeakageLabel",
        palette=palette_label,
        showfliers=False,        # cleaner for thesis
        width=0.65
    )

    # Fix duplicated legends (box + strip)
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles[:2], labels[:2], title="Condition")

    plt.title(f"{feat} – Healthy vs Leakage per Kit")
    plt.xlabel("Kit / Source")
    plt.ylabel(feat)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df_plot['y_jitter'] = rng.normal(0, 0.02, size=len(df_plot))
# df_filtered = df_plot[(df_plot['Max_pressure_pipe'] < 0.8) & (df_plot['WV_bin'] == 1)].copy()
# df_filtered = df_plot[(df_plot['EmergencyBrake_action'] == 0) & (df_plot['WV_bin'] == 1)].copy()
df_filtered = df_plot[
    (df_plot['DataSource'] == 0) & 
    (df_plot['WV_bin'].isin([1]))
].copy()
# df_filtered = df_plot[(df_plot['DataSource'] == 0) & (df_plot['WV_bin'] == 1)].copy()
# df_filtered = df_plot[(df_plot['WV_bin'] == 1)].copy()
# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df_plot['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df_plot['label'].unique()))

# Map categories to colors
source_codes = df_filtered['Source'].astype('category').cat.codes
label_codes  = df_filtered['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filtered['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filtered['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")
axes[1].set_xlim(-1, 2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    Plot Total_power_efficiency against each selected feature.
    Creates N separate figures (one per feature).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data containing 'Total_power_efficiency', 'label', and selected features.
    features : list of str
        List of feature column names to plot against Total_power_efficiency.
    base_width : int, optional
        Width of each figure (default=7).
    base_height : int, optional
        Height of each figure (default=5).
    x_limits : tuple (min, max), optional
        Limits for the X-axis (Total_power_efficiency).
    """
    # Masks for labels
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    
    for feature in features:
        plt.figure(figsize=(base_width, base_height))
        
        plt.scatter(
            df.loc[mask_0, 'Total_power_efficiency'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='0'
        )
        plt.scatter(
            df.loc[mask_1, 'Total_power_efficiency'],
            df.loc[mask_1, feature],
            color='red', alpha=0.7, label='1'
        )
        
        plt.xlabel("Total_power_efficiency")
        plt.ylabel(feature)
        plt.title(f"Total_power_efficiency vs {feature}")
        plt.legend(title='label', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Apply X-axis limits if provided
        if x_limits is not None:
            plt.xlim(x_limits)
        
        plt.tight_layout()
        plt.show()

# Example usage:
df_filtered = df_plot[df_plot['WV_bin'] == 1]

selected_features = ["Total_power_delay", "WV_MeanPressure","Total_energy_efficiency","Std_delay_exp","Mean_delay_exp"]

# Limit X-axis between 0 and 100
plot_efficiency_vs_features(df_filtered, selected_features, x_limits=(0, 10))


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    For each feature in `features`, create ONE figure with two subplots:

    - LEFT: Total_power_efficiency vs feature, colored by binary df['label'] (0 / 1)
    - RIGHT: Total_power_efficiency vs feature, colored by df['Malfunction'] groups:
        * C, E, G -> green
        * D, F     -> purple
        * 0        -> blue
        * A, B     -> cyan
    """
    # Masks for labels (left plot)
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1

    # Convenience: Malfunction column (assumed to exist)
    mal = df['Malfunction']

    # Define malfunction groups (right plot)
    group_defs = {
        "Healthy / 0":            (mal == 0) | (mal == '0'),
        "2.5 mm efflux (C,E,G)":   mal.isin(['C', 'E', 'G']),
        "1 mm efflux (D,F)":     mal.isin(['D', 'F']),
        "Auxilarry Leakage (A,B)":   mal.isin(['A', 'B']),
    }

    group_colors = {
        "Healthy / 0":            'blue',
        "2.5 mm efflux (C,E,G)":   'green',
        "1 mm efflux (D,F)":     'purple',
        "Auxilarry Leakage (A,B)":   'cyan',
    }

    for feature in features:
        fig, axes = plt.subplots(
            1, 2,
            figsize=(2 * base_width, base_height),
            sharex=True,  # same Total_power_efficiency scale
            sharey=False  # y scale may differ feature to feature
        )

        # -------------------------------------------------
        # LEFT: binary label (0/1) as in your original code
        # -------------------------------------------------
        ax_left = axes[0]

        ax_left.scatter(
            df.loc[mask_0, 'Total_power_efficiency'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='label = 0'
        )
        ax_left.scatter(
            df.loc[mask_1, 'Total_power_efficiency'],
            df.loc[mask_1, feature],
            color='red', alpha=0.7, label='label = 1'
        )

        ax_left.set_xlabel("Total_power_efficiency")
        ax_left.set_ylabel(feature)
        # ax_left.set_title(f"Total_power_efficiency vs {feature}")
        ax_left.legend(loc='best')

        if x_limits is not None:
            ax_left.set_xlim(x_limits)

        # -------------------------------------------------
        # RIGHT: malfunction-based coloring
        # -------------------------------------------------
        ax_right = axes[1]

        # plot each group with its own color
        for group_name, group_mask in group_defs.items():
            if group_mask.any():
                ax_right.scatter(
                    df.loc[group_mask, 'Total_power_efficiency'],
                    df.loc[group_mask, feature],
                    color=group_colors[group_name],
                    alpha=0.7,
                    label=group_name
                )

        ax_right.set_xlabel("Total_power_efficiency")
        ax_right.set_ylabel(feature)
        # ax_right.set_title(f"Malfunction groups: Total_power_efficiency vs {feature}", fontsize=10)
        ax_right.legend(loc='upper right', fontsize=12)


        if x_limits is not None:
            ax_right.set_xlim(x_limits)
        fig.suptitle(f"Total_power_efficiency vs {feature}", fontsize=16)
        fig.tight_layout()
        plt.show()


# Example usage:
df_filtered = df_plot[df_plot['WV_bin'] == 1]
plot_features = [
    "Total_power_delay",
    "WV_MeanPressure",
    "Total_energy_efficiency",
    "Std_delay_exp",
    "Total_energy_delay"
]

plot_efficiency_vs_features(df_filtered, plot_features, x_limits=(0, 10))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df_filt = df_plot[df_plot["WV_bin"] == 1].copy()
df_filt['y_jitter'] = rng.normal(0, 0.02, size=len(df_filt))

# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df_filt['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filt['Source'].astype('category').cat.codes
label_codes  = df_filt['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filt['Buildup_end_pressure_delay'],
    df_filt['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filt['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filt['Buildup_end_pressure_delay'],
    df_filt['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filt['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")

plt.tight_layout()
plt.show()


# Algorithm 1 - 2 Features

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING TRAIN TEST SPLIT AND FEATURE SELECTION 
# ============================================================================
selected_features = ['Total_power_efficiency','Std_delay_exp'] 

[X_train, X_test, y_train, y_test, mal_train, mal_test, X_train_healthy] = preprocess_data(df, selected_features, test_size=0.2,random_state=42)
X_train.head()

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
## matplotlib inline
matplotlib.style.use('fivethirtyeight')
df_plot = df.copy()
x = X_train[selected_features]
robust_df = pd.DataFrame(preprocessing.RobustScaler().fit_transform(x), 
                         columns=selected_features)
standard_df = pd.DataFrame(preprocessing.StandardScaler().fit_transform(x), 
                           columns=selected_features)
minmax_df = pd.DataFrame(preprocessing.MinMaxScaler().fit_transform(x), 
                         columns=selected_features)

fig, axes = plt.subplots(ncols=4, figsize=(20, 5))
datasets = [x, robust_df, standard_df, minmax_df]
titles = ['Before Scaling', 'Robust Scaling', 'Standard Scaling', 'Min-Max Scaling']
colors = ['r', 'b', 'g', 'cyan', 'orange', 'purple']  # extend for more features

for ax, df_plot, title in zip(axes, datasets, titles):
    ax.set_title(title)
    for i, feature in enumerate(df_plot.columns):
        sns.kdeplot(df_plot[feature], ax=ax, color=colors[i % len(colors)], label=feature)
    ax.legend()
plt.show()

In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

# Align y_train_healthy with X_train_healthy
y_train_healthy = y_train.loc[X_train_healthy.index]

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train, X_test, X_train_healthy)

### Downsampling Method to select only Borderline Points

reference: A scalable fuzzy support vector machine for fault detection in
transportation systems
Jie Liu a, Enrico Zio (2018)

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def reverse_nearest_neighbors_outlier_detection(X, y, k=5):
    """
    RNN outlier detection that preserves original indices.

    Returns
    -------
    X_cleaned : same type as X
    y_cleaned : same type as y
    outlier_indices : np.ndarray (original indices of outliers)
    kept_indices : np.ndarray (original indices of kept points)
    """
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)
    n_samples = len(X_arr)

    if hasattr(X, "index"):
        original_idx = X.index.to_numpy()
    else:
        original_idx = np.arange(n_samples)

    classes = np.unique(y_arr)
    reverse_neighbor_count = np.zeros(n_samples)

    for class_label in classes:
        class_indices = np.where(y_arr == class_label)[0]
        X_class = X_arr[class_indices]

        if len(class_indices) <= k:
            reverse_neighbor_count[class_indices] = 1
            continue

        nbrs = NearestNeighbors(n_neighbors=min(k + 1, len(X_class)))
        nbrs.fit(X_class)
        _, indices = nbrs.kneighbors(X_class)

        for i, neighbors in enumerate(indices):
            for neighbor_idx in neighbors[1:]:     # skip self
                actual_idx = class_indices[neighbor_idx]
                reverse_neighbor_count[actual_idx] += 1

    outlier_mask = reverse_neighbor_count == 0
    clean_mask   = ~outlier_mask

    outlier_indices = original_idx[outlier_mask]
    kept_indices    = original_idx[clean_mask]

    if hasattr(X, "iloc"):
        X_cleaned = X.iloc[clean_mask].copy()
    else:
        X_cleaned = X_arr[clean_mask]

    if hasattr(y, "iloc"):
        y_cleaned = y.iloc[clean_mask].copy()
    else:
        y_cleaned = y_arr[clean_mask]

    print(f"Total samples: {n_samples}")
    print(f"Outliers detected: {len(outlier_indices)}")
    print(f"Samples after cleaning: {len(X_cleaned)}")

    return X_cleaned, y_cleaned, outlier_indices, kept_indices


def knn_borderline_selection(X, y, indices=None, k=5, verbose=True):
    """
    Select borderline points using k-Nearest Neighbors, preserving original indices.

    Parameters
    ----------
    X : array-like or DataFrame
    y : array-like or Series
    indices : array-like or None
        Original indices of the rows in X,y. If None, use 0..n-1.
    k : int
        Number of neighbors.

    Returns
    -------
    X_borderline : same type as X
    y_borderline : same type as y
    borderline_indices : np.ndarray (original indices of BORDERLINE points)
    """
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)
    n_samples = len(X_arr)

    if indices is None:
        indices = np.arange(n_samples)
    else:
        indices = np.asarray(indices)

    nbrs = NearestNeighbors(n_neighbors=min(k + 1, n_samples))
    nbrs.fit(X_arr)
    distances, nn_indices = nbrs.kneighbors(X_arr)

    borderline_mask = np.zeros(n_samples, dtype=bool)

    for i in range(n_samples):
        neighbor_idx = nn_indices[i][1:k+1]          # skip self
        neighbor_labels = y_arr[neighbor_idx]
        if len(np.unique(neighbor_labels)) > 1:
            borderline_mask[i] = True
            if verbose and i < 3:
                print(f"Point {i} (class={y_arr[i]}): {neighbor_labels} → BORDERLINE")
        else:
            if verbose and i < 3:
                print(f"Point {i} (class={y_arr[i]}): {neighbor_labels} → NOT BORDERLINE")

    borderline_indices = indices[borderline_mask]

    if hasattr(X, "iloc"):
        X_borderline = X.iloc[borderline_mask].copy()
    else:
        X_borderline = X_arr[borderline_mask]

    if hasattr(y, "iloc"):
        y_borderline = y.iloc[borderline_mask].copy()
    else:
        y_borderline = y_arr[borderline_mask]

    if verbose:
        print(f"\nTotal samples: {n_samples}")
        print(f"Borderline points selected: {len(X_borderline)}")
        print(f"Data reduction: {100 * (1 - len(X_borderline)/n_samples):.1f}% removed")

    return X_borderline, y_borderline, borderline_indices

def complete_preprocessing(X, y, k_outlier=5, k_borderline=5):
    """
    Outlier removal (RNN) + borderline selection (KNN),
    with full index tracking.
    """
    print("=" * 50)
    print("Step 1: Outlier Detection using RNN")
    print("=" * 50)
    X_clean, y_clean, out_idx, kept_idx = reverse_nearest_neighbors_outlier_detection(
        X, y, k=k_outlier
    )

    print("\n" + "=" * 50)
    print("Step 2: Borderline Point Selection using KNN")
    print("=" * 50)
    X_final, y_final, final_idx = knn_borderline_selection(
        X_clean, y_clean, indices=kept_idx, k=k_borderline
    )

    print("\n" + "=" * 50)
    print(f"Data reduction: {len(X)} → {len(X_final)} "
          f"({100 * len(X_final) / len(X):.2f}%)")
    print("=" * 50)

    # out_idx: original outliers
    # kept_idx: original non-outliers (after RNN)
    # final_idx: original indices of borderline, non-outlier points
    return X_final, y_final, out_idx, kept_idx, final_idx



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_outlier_removal_2d(X, y, outlier_indices, f0=0, f1=1):
    """
    Visualize RNN outlier removal in 2D using two features.

    Parameters
    ----------
    X : array-like or pandas.DataFrame, shape (n_samples, n_features)
        Original data (before outlier removal).
    y : array-like or pandas.Series, shape (n_samples,)
        Class labels.
    outlier_indices : array-like
        ORIGINAL indices of outliers (as returned by
        reverse_nearest_neighbors_outlier_detection).
    f0, f1 : int
        Indices of features to plot on x and y axis.
    """
    # ------------------------------------------------------------
    # Handle pandas vs numpy, and keep track of original indices
    # ------------------------------------------------------------
    if hasattr(X, "values"):  # pandas DataFrame
        X_arr = np.asarray(X.values)
        idx_arr = np.asarray(X.index)        # original index
        feat_names = getattr(X, "columns", None)
    else:  # plain numpy array or similar
        X_arr = np.asarray(X)
        idx_arr = np.arange(len(X_arr))
        feat_names = None

    y_arr = np.asarray(y)
    outlier_indices = np.asarray(outlier_indices)

    n_samples = len(X_arr)

    # ------------------------------------------------------------
    # Build mask: outlier_indices are ORIGINAL indices
    # ------------------------------------------------------------
    # For each row, check if its original index is in outlier_indices
    outlier_mask = np.isin(idx_arr, outlier_indices)
    clean_mask = ~outlier_mask

    # Safety check in case someone passes positional indices by mistake:
    # if no overlap at all but all outlier_indices < n_samples, you might
    # want to treat them as positions instead. Uncomment if needed:
    # if not outlier_mask.any() and outlier_indices.max() < n_samples:
    #     outlier_mask = np.zeros(n_samples, dtype=bool)
    #     outlier_mask[outlier_indices] = True
    #     clean_mask = ~outlier_mask

    x0, x1 = X_arr[:, f0], X_arr[:, f1]

    # Label axes with feature names if available
    if feat_names is not None:
        label_x = feat_names[f0]
        label_y = feat_names[f1]
    else:
        label_x = f"Feature {f0}"
        label_y = f"Feature {f1}"

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ---------------- BEFORE ----------------
    ax = axes[0]
    # non-outliers (all classes)
    ax.scatter(
        x0[clean_mask],
        x1[clean_mask],
        c="lightgray",
        alpha=0.6,
        label="Non-outliers"
    )
    # outliers highlighted
    ax.scatter(
        x0[outlier_mask],
        x1[outlier_mask],
        c="red",
        edgecolor="k",
        alpha=0.9,
        label="Detected outliers"
    )
    ax.set_title("Before K-nearest Outlier Removal")
    ax.set_xlabel(label_x)
    ax.set_ylabel(label_y)
    ax.legend()

    # ---------------- AFTER ----------------
    ax = axes[1]
    sc = ax.scatter(
        x0[clean_mask],
        x1[clean_mask],
        c=y_arr[clean_mask],
        cmap="coolwarm",
        alpha=0.7
    )
    ax.set_title("After K-nearest Outlier Removal")
    ax.set_xlabel(label_x)
    ax.set_ylabel(label_y)

    # Optional: colorbar for class labels
    # plt.colorbar(sc, ax=ax, label="Class label")

    plt.tight_layout()
    plt.show()


In [ ]:
from collections import Counter
# Example usage
y_ds = y_train.values
X_ds = X_train_scaled
k_outlier = 10
print(f"Original dataset: {X_ds.shape[0]} samples")
print(f"Class distribution: {Counter(y_ds)}")
print()

# Apply complete preprocessing
X_border, y_border, out_idx, kept_idx, final_idx = complete_preprocessing(
    X_ds, y_ds, k_outlier=10, k_borderline=100
)

print(f"\nFinal class distribution: {Counter(y_border)}")

In [ ]:
X_train_clean, y_train_clean, out_idx, kept_idx = reverse_nearest_neighbors_outlier_detection(
    X_train_scaled, y_train, k=10
)
# X_train_clean and y_train_clean keep the original indices from df_filt
# out_idx tells you which original rows were flagged as outliers.
# Malfunction labels for the *clean* training points:
mal_train_clean = mal_train.iloc[kept_idx]     # same original indices
# Then for visualization (before/after):
plot_outlier_removal_2d(X_train_scaled, y_train, out_idx, f0=0, f1=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ----------------------------------------------------
# 1. PCA → reduce to 2 components for visualization
# ----------------------------------------------------
pca = PCA(n_components=2)
X_before_2d = pca.fit_transform(X_ds)
X_after_2d  = pca.transform(X_border)

y_before = y_ds
y_after  = y_border

# ----------------------------------------------------
# 2. Determine a common axis scale for both plots
# ----------------------------------------------------
all_points = np.vstack([X_before_2d, X_after_2d])
xmin, xmax = all_points[:,0].min(), all_points[:,0].max()
ymin, ymax = all_points[:,1].min(), all_points[:,1].max()

# Add small padding for readability
padding_x = 0.05 * (xmax - xmin)
padding_y = 0.05 * (ymax - ymin)

xmin -= padding_x; xmax += padding_x
ymin -= padding_y; ymax += padding_y

# ----------------------------------------------------
# 3. Plot with SAME axis limits
# ----------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# BEFORE
axes[0].scatter(
    X_before_2d[y_before == 0, 0],
    X_before_2d[y_before == 0, 1],
    alpha=0.5, label="Healthy (0)", c="blue"
)
axes[0].scatter(
    X_before_2d[y_before == 1, 0],
    X_before_2d[y_before == 1, 1],
    alpha=0.7, label="Faulty (1)", c="red"
)
axes[0].set_title("Before Preprocessing")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_xlim(xmin, xmax)
axes[0].set_ylim(ymin, ymax)
axes[0].legend()

# AFTER
axes[1].scatter(
    X_after_2d[y_after == 0, 0],
    X_after_2d[y_after == 0, 1],
    alpha=0.5, label="Healthy (0)", c="blue"
)
axes[1].scatter(
    X_after_2d[y_after == 1, 0],
    X_after_2d[y_after == 1, 1],
    alpha=0.7, label="Faulty (1)", c="red"
)
axes[1].set_title("After Preprocessing\n(outlier removal + borderline sampling)")
axes[1].set_xlabel("PC1")
axes[1].set_xlim(xmin, xmax)
axes[1].set_ylim(ymin, ymax)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------
# Select real features for plotting
# ----------------------------------------------------
f0_before = X_ds[:, 0]
f1_before = X_ds[:, 1]

f0_after  = X_border[:, 0]
f1_after  = X_border[:, 1]

y_before = y_ds
y_after  = y_border

# ----------------------------------------------------
# Define common axis limits
# ----------------------------------------------------
all_x = np.concatenate([f0_before, f0_after])
all_y = np.concatenate([f1_before, f1_after])

xmin, xmax = all_x.min(), all_x.max()
ymin, ymax = all_y.min(), all_y.max()

# add padding
pad_x = 0.05 * (xmax - xmin)
pad_y = 0.05 * (ymax - ymin)
xmin -= pad_x; xmax += pad_x
ymin -= pad_y; ymax += pad_y

# ----------------------------------------------------
# Plot
# ----------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# BEFORE
axes[0].scatter(
    f0_before[y_before == 0],
    f1_before[y_before == 0],
    alpha=0.5, label="Healthy (0)", c="blue"
)
axes[0].scatter(
    f0_before[y_before == 1],
    f1_before[y_before == 1],
    alpha=0.7, label="Faulty (1)", c="red"
)
axes[0].set_title("Before Preprocessing")
axes[0].set_xlabel("Feature 0")
axes[0].set_ylabel("Feature 1")
axes[0].set_xlim(xmin, xmax)
axes[0].set_ylim(ymin, ymax)
axes[0].legend()

# AFTER
axes[1].scatter(
    f0_after[y_after == 0],
    f1_after[y_after == 0],
    alpha=0.5, label="Healthy (0)", c="blue"
)
axes[1].scatter(
    f0_after[y_after == 1],
    f1_after[y_after == 1],
    alpha=0.7, label="Faulty (1)", c="red"
)
axes[1].set_title("After Preprocessing")
axes[1].set_xlabel("Feature 0")
axes[1].set_xlim(xmin, xmax)
axes[1].set_ylim(ymin, ymax)
axes[1].legend()

plt.tight_layout()
plt.show()


### SMOTE

#### Differences between SMOTE Function

In [ ]:
from collections import Counter
import pandas as pd
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, KMeansSMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# ---------------------------------------------------------
# BASE DATA
# ---------------------------------------------------------
X = X_train_clean
y = y_train_clean
sampling_ratio = 0.8

# Wrap in DataFrame for consistent columns
if isinstance(X, pd.DataFrame):
    feature_names = X.columns.tolist()
    X_df = X.copy()
else:
    feature_names = [f"feat_{i}" for i in range(X.shape[1])]
    X_df = pd.DataFrame(X, columns=feature_names)

y_ser = pd.Series(y, name="label")

# ---------------------------------------------------------
# DICTIONARY OF ALL SAMPLERS
# ---------------------------------------------------------
oversamplers = {
    "SMOTE": SMOTE(sampling_strategy=sampling_ratio, random_state=42),
    "BorderlineSMOTE": BorderlineSMOTE(sampling_strategy=sampling_ratio, random_state=42),
    "SMOTE-Tomek": SMOTETomek(sampling_strategy=sampling_ratio, random_state=42),
    "SMOTE-ENN": SMOTEENN(sampling_strategy=sampling_ratio, random_state=42),
    "ADASYN": ADASYN(sampling_strategy=sampling_ratio, random_state=42),
    "KMeans-SMOTE": KMeansSMOTE(sampling_strategy=sampling_ratio, random_state=42)
}

# ---------------------------------------------------------
# CREATE THE MAIN DICTIONARY OF ALL RESULTS
# ---------------------------------------------------------
resampled_data = {}   # MASTER DICT
summary_rows = []

for name, sampler in oversamplers.items():
    pipeline = Pipeline([
        ("sampler", sampler)
    ])
    
    X_res, y_res = pipeline.fit_resample(X_df, y_ser)

    # store in a structured dict
    resampled_data[name] = {
        "X": pd.DataFrame(X_res, columns=feature_names),
        "y": pd.Series(y_res, name="label"),
        "sampler": sampler
    }

    # Compute useful statistics
    cnt = Counter(y_res)
    summary_rows.append({
        "Sampler": name,
        "Total_samples": len(y_res),
        "Class_0_count": cnt.get(0, 0),
        "Class_1_count": cnt.get(1, 0),
        "Minority_class": min(cnt, key=cnt.get),
        "Majority_class": max(cnt, key=cnt.get),
        "Imbalance_ratio": max(cnt.values()) / min(cnt.values())
    })

# Add original dataset for comparison
cnt0 = Counter(y_ser)
summary_rows.insert(0, {
    "Sampler": "Original",
    "Total_samples": len(y_ser),
    "Class_0_count": cnt0.get(0, 0),
    "Class_1_count": cnt0.get(1, 0),
    "Minority_class": min(cnt0, key=cnt0.get),
    "Majority_class": max(cnt0, key=cnt0.get),
    "Imbalance_ratio": max(cnt0.values()) / min(cnt0.values())
})

resample_summary_df = pd.DataFrame(summary_rows)
resample_summary_df


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Ensure X is a DataFrame
# ------------------------------------------------------------
if isinstance(X_train_clean, pd.DataFrame):
    X_orig = X_train_clean.copy()
    feature_names = X_orig.columns.tolist()
else:
    X_orig = pd.DataFrame(X_train_clean)
    feature_names = X_orig.columns.tolist()

y_orig = pd.Series(y_train_clean, name="label")


# ------------------------------------------------------------
# FUNCTION: Scatterplot before vs after SMOTE/ADASYN/etc.
# ------------------------------------------------------------
def scatter_compare_by_index(
    X_orig, y_orig, 
    X_res, y_res,
    idx_x, idx_y, 
    sampler_name,
    selected_features = None
):
    if selected_features is not None:
        feat_x = selected_features[idx_x]
        feat_y = selected_features[idx_y]
    else:
        feat_x = feature_names[idx_x]
        feat_y = feature_names[idx_y]
        
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # BEFORE (Original)
    axes[0].scatter(
        X_orig.iloc[:, idx_x],
        X_orig.iloc[:, idx_y],
        c=y_orig, cmap="coolwarm",
        alpha=0.5, s=20
    )
    axes[0].set_title(
        f"Original Scaled –",
        fontsize=16, fontweight="bold"
    )
    axes[0].set_xlabel(feat_x, fontsize=14)
    axes[0].set_ylabel(feat_y, fontsize=14)
    axes[0].tick_params(labelsize=12)
    axes[0].set_xlim([-2, 2])

    # AFTER (Resampled)
    axes[1].scatter(
        X_res.iloc[:, idx_x],
        X_res.iloc[:, idx_y],
        c=y_res, cmap="coolwarm",
        alpha=0.5, s=20
    )
    axes[1].set_title(
        f"{sampler_name}",
        fontsize=16, fontweight="bold"
    )
    axes[1].set_xlabel(feat_x, fontsize=14)
    axes[1].set_ylabel(feat_y, fontsize=14)
    axes[1].tick_params(labelsize=12)
    axes[1].set_xlim([-2, 2])
    fig.tight_layout(rect=[0, 0, 1, 0.95])  # Leave space for titles
    plt.show()



# ------------------------------------------------------------
# CHOOSE FEATURE INDEX PAIRS TO PLOT
# ------------------------------------------------------------
feature_pairs_idx = [
    (0,1 ),     # Example TPE and TPD
    # Add more pairs if needed
]
# selected_features = ['Std_delay_exp', 'Total_power_efficiency','Total_power_delay','Max_pressure_pipe'] 

# ------------------------------------------------------------
# LOOP THROUGH ALL RESAMPLERS AND PLOT SCATTERS
# resampled_data[name] = {"X": df, "y": series, "sampler": obj}
# ------------------------------------------------------------
for sampler_name, data_dict in resampled_data.items():

    # Skip the original dataset if you inserted it manually
    if sampler_name.lower() == "original":
        continue

    X_res_df = data_dict["X"]
    y_res_ser = data_dict["y"]

    print(f"\n=== Scatterplots for {sampler_name} ===")

    for idx_x, idx_y in feature_pairs_idx:
        scatter_compare_by_index(
            X_orig, y_orig,
            X_res_df, y_res_ser,
            idx_x, idx_y,
            sampler_name,
            selected_features=selected_features
        )


## Model Definition and Helper Function

### Defining function for Model used, Train Model, and Cross Validations

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE


def get_all_models(X_train):
    """
    Return ONLY supervised models intended for training on SMOTE-resampled data.
    """
    n_features = X_train.shape[1]

    models = {
        'supervised_smote': {
            'Random Forest (SMOTE)': RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                max_depth=n_features,
                n_jobs=-1
            ),

            'XGBoost (SMOTE)': XGBClassifier(
                n_estimators=200,
                random_state=42,
                max_depth=n_features,
                eval_metric='logloss'
            ),

            'Logistic Regression (SMOTE)': LogisticRegression(
                C=0.1,
                random_state=42,
                max_iter=1000,
                solver='liblinear'
            ),

            'Decision Tree (SMOTE)': DecisionTreeClassifier(
                random_state=42,
                max_depth=n_features,
                max_features=n_features,
                min_samples_leaf=2,
                min_samples_split=2
            ),

            'KNN (SMOTE)': KNeighborsClassifier(
                n_neighbors=15,
                weights='distance',
                metric='euclidean',
                p=1
            ),

            'SVM (SMOTE)': SVC(
                kernel='linear',        # common choice, can be 'linear' too
                C=1.0,               # regularization strength
                gamma='scale',       # auto scaling of kernel coefficient
                probability=True,    # enables predict_proba for ROC/AUC
                random_state=42
            ),
            
            'SVM_rbf (SMOTE)': SVC(
                kernel='rbf',        # common choice, can be 'linear' too
                C=1.0,               # regularization strength
                gamma='scale',       # auto scaling of kernel coefficient
                probability=True,    # enables predict_proba for ROC/AUC
                random_state=42
            )
        }
    }

    return models

In [ ]:
def train_all_models(X_train, y_train, model_group='supervised_smote'):
    """
    Train a group of models on the *already prepared* training data
    (e.g., after SMOTE / ADASYN / KMeansSMOTE, or even original).

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training features (can already be oversampled and scaled).
    y_train : array-like or Series
        Training labels (aligned with X_train).
    model_group : str
        Which group from get_all_models() to use, e.g.:
        - 'supervised'        : if you trained on original data
        - 'supervised_smote'  : if you conceptually group these as SMOTE-based models
        - any other key you defined in get_all_models()

    Returns
    -------
    trained_models : dict
        { model_name: {'model': estimator, 'type': model_group, 'trained': True} }
    """

    models = get_all_models(X_train)          # your existing function
    if model_group not in models:
        raise ValueError(
            f"Model group '{model_group}' not found in get_all_models(). "
            f"Available groups: {list(models.keys())}"
        )

    model_dict = models[model_group]
    trained_models = {}

    print("\n" + "="*60)
    print(f"TRAINING MODELS ({model_group}) ON PROVIDED DATA")
    print("="*60)
    print(f"  - X_train shape: {X_train.shape}")
    print(f"  - y_train distribution: "
          f"0={sum(np.array(y_train)==0)}, 1={sum(np.array(y_train)==1)}")

    for name, model in model_dict.items():
        print(f"  - Training {name}...")
        model.fit(X_train, y_train)
        trained_models[name] = {
            'model': model,
            'type': model_group,
            'trained': True
        }

    print("\nAll models trained successfully!")
    return trained_models


### CROSS VALIDATION METHOD


In [ ]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, KMeansSMOTE
from imblearn.combine import SMOTETomek, SMOTEENN


def get_oversampler(name="SMOTE", sampling_ratio=None, random_state=42):
    """
    Return an oversampler object from imbalanced-learn, based on a string name.

    Parameters
    ----------
    name : str
        One of:
        - "SMOTE"
        - "BorderlineSMOTE"
        - "ADASYN"
        - "KMeans-SMOTE"
        - "SMOTE-Tomek"
        - "SMOTE-ENN"

    sampling_ratio : float or dict or 'auto' or None
        Passed to sampling_strategy.
        Example: 0.5 -> minority = 50% of majority class.

    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    oversampler : object with fit_resample(X, y)
    """
    if sampling_ratio is None:
        sampling_strategy = 'auto'
    else:
        sampling_strategy = sampling_ratio

    name = name.lower()

    if name == "smote":
        return SMOTE(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name == "borderlinesmote":
        return BorderlineSMOTE(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name == "adasyn":
        return ADASYN(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name in ("kmeans-smote", "kmeanssmote"):
        return KMeansSMOTE(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name in ("smote-tomek", "smotetomek"):
        return SMOTETomek(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name in ("smote-enn", "smoteenn"):
        return SMOTEENN(sampling_strategy=sampling_strategy, random_state=random_state)

    else:
        raise ValueError(
            f"Unknown oversampler name '{name}'. "
            "Use one of: 'SMOTE', 'BorderlineSMOTE', 'ADASYN', "
            "'KMeans-SMOTE', 'SMOTE-Tomek', 'SMOTE-ENN'."
        )


In [ ]:
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, cross_val_predict
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve
)
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
import pandas as pd
import numpy as np


def cv_metrics_table(
    trained_models,
    X,
    y,
    use_smote_in_cv=True,
    oversampler_name="SMOTE",
    sampling_ratio=None,
    include_types=('supervised_smote',)
):
    """
    Cross-validated metrics table with per-model threshold tuning
    based on out-of-fold (OOF) predicted probabilities.

    Option A (recommended): use_smote_in_cv=True
    - X, y should be the ORIGINAL imbalanced training set.
    - The chosen oversampler (SMOTE or variant) is applied inside each CV fold.

    Parameters
    ----------
    trained_models : dict
        { model_name: {'model': estimator, 'type': str, ...} }

    X, y : array-like
        Data for CV. For Option A:
        - X = original scaled features
        - y = original labels (imbalanced)

    use_smote_in_cv : bool
        If True, wrap each model in [oversampler -> classifier] ImbPipeline inside CV.

    oversampler_name : str
        Which oversampling method to use in CV:
        - "SMOTE"
        - "BorderlineSMOTE"
        - "ADASYN"
        - "KMeans-SMOTE"
        - "SMOTE-Tomek"
        - "SMOTE-ENN"

    sampling_ratio : float or None
        Passed to sampling_strategy of the oversampler.
        Example: 0.5 -> minority = 50% of majority.

    include_types : tuple of str
        Only models whose entry['type'] is in include_types are evaluated.
        Typical: ('supervised_smote',) or ('supervised', 'supervised_smote').

    Returns
    -------
    df : pd.DataFrame
        Metrics per model (Precision, Recall, F1, ROC-AUC, confusion matrix entries)
        at the threshold that maximizes F1 on OOF predictions.
    """

    X = np.asarray(X)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    # cv = LeaveOneOut()
    rows = []

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry.get('type', None)

        if mtype not in include_types:
            continue

        # ---------------------------------------------------------
        # 1. Build estimator for CV
        # ---------------------------------------------------------
        if use_smote_in_cv:
            oversampler = get_oversampler(
                name=oversampler_name,
                sampling_ratio=sampling_ratio,
                random_state=42
            )

            estimator = ImbPipeline([
                ('oversampler', oversampler),
                ('clf', clone(base_model))
            ])
        else:
            estimator = clone(base_model)

        # OOF predicted probabilities for the positive class
        y_pred_proba = cross_val_predict(
            estimator, X, y,
            cv=cv, method='predict_proba'
        )[:, 1]

        # # ---------------------------------------------------------
        # # 2. THRESHOLD TUNING (on OOF probabilities)
        # # ---------------------------------------------------------
        # precision_arr, recall_arr, thresholds = precision_recall_curve(y, y_pred_proba)

        # # F1 for each threshold; thresholds has len = len(precision_arr) - 1
        # f1_scores = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-8)
        # best_idx = np.argmax(f1_scores[:-1])  # last element has no corresponding threshold
        # best_threshold = thresholds[best_idx]
        best_threshold = 0.5

        # Convert probs to labels using tuned threshold
        y_pred = (y_pred_proba >= best_threshold).astype(int)

        # ---------------------------------------------------------
        # 3. METRICS WITH TUNED THRESHOLD
        # ---------------------------------------------------------
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

        precision = precision_score(y, y_pred, zero_division=0)
        recall    = recall_score(y, y_pred, zero_division=0)
        f1        = f1_score(y, y_pred, zero_division=0)
        rocauc    = roc_auc_score(y, y_pred_proba)

        rows.append({
            "Model": name,
            "Type": mtype,
            "Oversampler": oversampler_name if use_smote_in_cv else "None",
            "SamplingRatio": sampling_ratio,
            "BestThreshold": best_threshold,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "True Positives": tp,
            "False Positives": fp,
            "False Negatives": fn,
            "True Negatives": tn,
        })

    df = pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)
    return df


### Test Metrics Table

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def test_metrics_table(trained_models, cv_summary, X_test, y_test):
    """
    Evaluate tuned models on a TEST set using the thresholds
    previously selected via CV on the TRAIN set.

    Parameters
    ----------
    trained_models : dict
        Same structure you used before: {name: {'model': ..., 'type': ...}}.
        These models are already fitted on the TRAIN data.
    cv_summary : pd.DataFrame
        Output of cv_metrics_table on the TRAIN set, containing
        at least columns ['Model', 'BestThreshold'].
    X_test, y_test : array-like
        Held-out test data (no CV here).

    Returns
    -------
    pd.DataFrame
        Metrics on the test set for each model, using its CV-tuned threshold.
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry['type']

        if mtype not in ["supervised", "supervised_smote"]:
            # skip anomaly / unsupervised etc.
            continue

        # 1) Get the best threshold from TRAIN CV summary
        row = cv_summary.loc[cv_summary["Model"] == name]
        if row.empty:
            # model was not in cv_summary_1 (or got filtered out)
            continue
        best_threshold = row["BestThreshold"].values[0]

        # 2) Predict probabilities on the TEST set
        #    (for SMOTE pipelines, you should have refit the pipeline on TRAIN
        #     before building `trained_models`).
        y_proba = base_model.predict_proba(X_test)[:, 1]

        # 3) Apply threshold
        y_pred = (y_proba >= best_threshold).astype(int)

        # 4) Compute metrics
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)
        rocauc    = roc_auc_score(y_test, y_proba)

        rows.append({
            "Model": name,
            "Type": mtype,
            "BestThreshold": best_threshold,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "True Positives": tp,
            "False Positives": fp,
            "False Negatives": fn,
            "True Negatives": tn,
        })

    return pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def test_metrics_from_final(final_models, final_thresholds, X_test, y_test):
    """
    Evaluate already-fitted final models on TEST set, using
    per-model thresholds that were tuned on the TRAIN set.

    Parameters
    ----------
    final_models : dict
        {model_name: fitted_estimator}
        (e.g. final_models_S1, final_models_S2, ...)
    final_thresholds : dict
        {model_name: best_threshold_from_train}
        (e.g. final_thresholds_S1, ...)
    X_test, y_test : array-like
        Held-out test set.

    Returns
    -------
    pd.DataFrame
        Test metrics for each model.
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []

    for name, estimator in final_models.items():
        if name not in final_thresholds:
            continue

        thr = final_thresholds[name]

        # probabilities on TEST
        proba = estimator.predict_proba(X_test)[:, 1]
        y_pred = (proba >= thr).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        roc  = roc_auc_score(y_test, proba)

        rows.append({
            "Model": name,
            "BestThreshold_train": thr,
            "Precision_Test": prec,
            "Recall_Test": rec,
            "F1_Test": f1,
            "ROC-AUC_Test": roc,
            "TP_Test": tp,
            "FP_Test": fp,
            "FN_Test": fn,
            "TN_Test": tn,
        })

    return pd.DataFrame(rows).sort_values(
        by="F1_Test", ascending=False
    ).reset_index(drop=True)


### Model Hyperparameter Tuning

#### RANDOM FOREST 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_random_forest_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Random Forest on already-resampled data.
    No class_weight, no SMOTE inside CV.

    Parameters
    ----------
    X_res, y_res : resampled training data (e.g. from SMOTE, ADASYN)
    sampler_name : optional string, just for printing ("SMOTE", "ADASYN", ...)

    Returns
    -------
    best_estimator_, best_params_, best_score_
    """

    rf_base = RandomForestClassifier(
        random_state=42,
        max_depth=X_res.shape[1],
        n_jobs=-1
    )

    param_grid = {
        'n_estimators':      [100, 200],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 3, 5],
        'max_features':      ['sqrt', 'log2'],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    # cv = LeaveOneOut()

    print(f"\n===== Tuning Random Forest on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=rf_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best RF params:", grid.best_params_)
    print("Best RF CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### XGBOOST

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_xgboost_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for XGBoost on already-resampled data.
    Do NOT use scale_pos_weight here (class is already rebalanced).
    """

    # Base model (scale_pos_weight will be tuned)
    xgb_base = XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        n_estimators=200,
        max_depth=X_res.shape[1],
        # early_stopping_rounds=20,
        use_label_encoder=False,  # optional depending on xgboost version
    )

    # compute approximate neg/pos ratio:
    n_pos = (y_res == 1).sum()
    n_neg = (y_res == 0).sum()
    ratio = n_neg / max(n_pos, 1)

    param_grid = {
        "n_estimators": [100, 200],
        "learning_rate": [0.01, 0.1],
        # 'max_depth':         [2, 3, 4],
        "min_child_weight": [3, 4, 5],
        "subsample": [0.6, 0.8],
        "colsample_bytree": [0.6, 0.8],
        "gamma": [0.0, 0.1, 0.5],
        # tune around the empirical imbalance ratio
        "scale_pos_weight": [ratio],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    # cv = LeaveOneOut()

    print(f"\n===== Tuning XGBoost on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=xgb_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best XGB params:", grid.best_params_)
    print("Best XGB CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### LOGISTIC REGRESSION

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_logistic_regression_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Logistic Regression on already-resampled data.
    No class_weight, since oversampling already handled imbalance.
    """

    lr_base = LogisticRegression(
        solver='liblinear',   # supports l1 and l2
        max_iter=1000,
        random_state=42
    )

    param_grid = {
        'C': [0.001, 0.01, 0.1, 1],
        'penalty': ['l1'],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    # cv = LeaveOneOut()

    print(f"\n===== Tuning Logistic Regression on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=lr_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best LR params:", grid.best_params_)
    print("Best LR CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### KNN TUNING

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_knn_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for KNN on already-resampled data.
    """

    knn_base = KNeighborsClassifier()

    param_grid = {
        'n_neighbors': [5, 8, 10, 12],
        'weights': ['distance','uniform'],
        'metric': ['euclidean','manhattan','minkowski'],
        'p': [1]
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    # cv = LeaveOneOut()
    
    print(f"\n===== Tuning KNN on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=knn_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best KNN params:", grid.best_params_)
    print("Best KNN CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### DECISION TREE TUNING

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_decision_tree_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Decision Tree on already-resampled data.
    No class_weight here.
    """

    dt_base = DecisionTreeClassifier(
        class_weight="balanced", 
        max_depth=X_res.shape[1], 
        random_state=42
    )

    param_grid = {
        "criterion": ["gini", "entropy", "log_loss"],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [2, 3, 5],
        "max_features": ["sqrt", "log2", None],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    # cv = LeaveOneOut()

    grid = GridSearchCV(
        estimator=dt_base,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print(f"\n===== Tuning Decision Tree on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=dt_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best DT params:", grid.best_params_)
    print("Best DT CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### SVM Linear SMOTE

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_svm_linear_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for LINEAR SVM on already-resampled data.
    No class_weight, no resampling inside CV.

    Parameters
    ----------
    X_res, y_res : resampled training data (e.g. from SMOTE, ADASYN)
    sampler_name : optional string for printing ("SMOTE", "ADASYN", ...)

    Returns
    -------
    best_estimator_, best_params_, best_score_, cv_results_
    """

    svm_linear = SVC(
        kernel="linear",
        probability=True,
        random_state=42
    )

    param_grid = {
        "C": [0.01, 0.1, 1, 10]
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    print(f"\n===== Tuning LINEAR SVM on {sampler_name or 'resampled'} data =====")

    grid = GridSearchCV(
        estimator=svm_linear,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_res, y_res)

    print("Best Linear SVM params:", grid.best_params_)
    print("Best Linear SVM CV F1 :", grid.best_score_)

    return (
        grid.best_estimator_,
        grid.best_params_,
        grid.best_score_,
        grid.cv_results_
    )


#### SVM rbf SMOTE

In [ ]:
def tune_svm_rbf_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for RBF SVM on already-resampled data.
    No class_weight, no resampling inside CV.

    Parameters
    ----------
    X_res, y_res : resampled training data (e.g. from SMOTE, ADASYN)
    sampler_name : optional string for printing ("SMOTE", "ADASYN", ...)

    Returns
    -------
    best_estimator_, best_params_, best_score_, cv_results_
    """

    svm_rbf = SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )

    param_grid = {
        "C":     [0.1, 1, 10],
        "gamma": ["scale", 0.01, 0.1, 1]
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    print(f"\n===== Tuning RBF SVM on {sampler_name or 'resampled'} data =====")

    grid = GridSearchCV(
        estimator=svm_rbf,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_res, y_res)

    print("Best RBF SVM params:", grid.best_params_)
    print("Best RBF SVM CV F1 :", grid.best_score_)

    return (
        grid.best_estimator_,
        grid.best_params_,
        grid.best_score_,
        grid.cv_results_
    )


## Train the Model

In [ ]:
X_sm, y_sm = resampled_data["ADASYN"]["X"], resampled_data["ADASYN"]["y"]

trained_models = train_all_models(
    X_sm, y_sm,
    model_group='supervised_smote'   # or 'supervised' if you prefer
)

In [ ]:
cv_summary = cv_metrics_table(
    trained_models,          # e.g. from get_all_models()['supervised_smote'] but NOT yet refit
    X_sm,        # ORIGINAL scaled train data (imbalanced)
    y_sm,
    use_smote_in_cv=False,  # already applied before training
    oversampler_name="SMOTE",
    sampling_ratio=sampling_ratio,      # choose your ratio, or None for 'auto'
    include_types=('supervised_smote',)
)
cv_summary

Try the UNTUNNED MODEL on Test Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# List of supervised models you want to visualize
best_models = [
    "KNN (SMOTE)",
    "Logistic Regression (SMOTE)",
    "Decision Tree (SMOTE)",
    "Random Forest (SMOTE)",
    "XGBoost (SMOTE)",
]

# Convert test set to numpy
X = np.asarray(X_test_scaled)
y = np.asarray(y_test)

# --------------------------------------------------------
# LOOP THROUGH EACH MODEL AND CREATE A SEPARATE FIGURE
# --------------------------------------------------------
for model_name in best_models:

    # Retrieve model info
    model_info = trained_models[model_name]
    mtype = model_info['type']
    
    if mtype not in ('supervised', 'supervised_smote'):
        raise ValueError(f"{model_name} is not a supervised model.")

    base_model = model_info['model']

    # Retrieve best threshold from your CV summary table
    row = cv_summary.loc[
        cv_summary['Model'] == model_name
    ]
    if row.empty:
        raise ValueError(f"No CV summary row found for '{model_name}'.")

    best_threshold = row['BestThreshold'].iloc[0]

    # Predict on test set
    y_proba = base_model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= best_threshold).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    max_count = cm.max()  # ensures consistent color scale

    # --------------------------------------------------------
    # PLOT — ONE FIGURE PER MODEL
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(5, 4))

    im = ax.imshow(cm, cmap='coolwarm', vmin=0, vmax=max_count)

    ax.set_title(
        f"{model_name}\nThreshold = {best_threshold:.3f}",
        fontsize=12, pad=10
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax.set_yticklabels(['True Healthy', 'True Leakage'])
    ax.grid(False)
    # Annotate values
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > max_count / 2 else 'white'
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color=color, fontsize=11)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Count")

    plt.tight_layout()
    plt.show()


## Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract the two selected features
X_vis_left = X_train_clean   # assuming selected_features holds indices [i, j]
y_vis_left = y_train_clean

X_vis_right = X_test_scaled
y_vis_right = y_test
# Get the trained SVM model
svm_model = trained_models["SVM (SMOTE)"]["model"]

# Combine ranges from both datasets
x_min = min(X_vis_left[:, 0].min(), X_vis_right[:, 0].min()) - 1
x_max = max(X_vis_left[:, 0].max(), X_vis_right[:, 0].max()) + 1
y_min = min(X_vis_left[:, 1].min(), X_vis_right[:, 1].min()) - 1
y_max = max(X_vis_left[:, 1].max(), X_vis_right[:, 1].max()) + 1

# Use the same mesh grid for both plots
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))


# Decision function values (left)
Z_decision = svm_model.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z_decision = Z_decision.reshape(xx.shape)

# Predicted classes (right)
Z_predict = svm_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z_predict = Z_predict.reshape(xx.shape)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left plot
cf1 = axes[0].contourf(xx, yy, Z_decision, levels=50, cmap='coolwarm', alpha=0.7)
axes[0].contour(xx, yy, Z_decision, levels=[0], linewidths=2, colors='black')
axes[0].scatter(X_vis_left[:, 0], X_vis_left[:, 1], c=y_vis_left, cmap='coolwarm', edgecolors='k')
axes[0].set_xlim(x_min, x_max)
axes[0].set_ylim(y_min, y_max)

# Right plot
cf2 = axes[1].contourf(xx, yy, Z_predict, cmap='coolwarm', alpha=0.3)
axes[1].scatter(X_vis_right[:, 0], X_vis_right[:, 1], c=y_vis_right, cmap='coolwarm', edgecolors='k')
axes[1].set_xlim(x_min, x_max)
axes[1].set_ylim(y_min, y_max)
fig.suptitle("SVM Decision Function linear kernel", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract the two selected features
X_vis_left = X_train_clean   # assuming selected_features holds indices [i, j]
y_vis_left = y_train_clean

X_vis_right = X_test_scaled
y_vis_right = y_test
# Get the trained SVM model
svm_rbf_model = trained_models["SVM_rbf (SMOTE)"]["model"]

# Combine ranges from both datasets
x_min = min(X_vis_left[:, 0].min(), X_vis_right[:, 0].min()) - 1
x_max = max(X_vis_left[:, 0].max(), X_vis_right[:, 0].max()) + 1
y_min = min(X_vis_left[:, 1].min(), X_vis_right[:, 1].min()) - 1
y_max = max(X_vis_left[:, 1].max(), X_vis_right[:, 1].max()) + 1

# Use the same mesh grid for both plots
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))


# Decision function values (left)
Z_decision = svm_model.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z_decision = Z_decision.reshape(xx.shape)

# Predicted classes (right)
Z_predict = svm_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z_predict = Z_predict.reshape(xx.shape)
# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Left plot
cf1 = axes[0].contourf(xx, yy, Z_decision, levels=50, cmap='coolwarm', alpha=0.7)
axes[0].contour(xx, yy, Z_decision, levels=[0], linewidths=2, colors='black')
axes[0].scatter(X_vis_left[:, 0], X_vis_left[:, 1], c=y_vis_left, cmap='coolwarm', edgecolors='k')
axes[0].set_xlim(x_min, x_max)
axes[0].set_ylim(y_min, y_max)
# Right plot
cf2 = axes[1].contourf(xx, yy, Z_predict, cmap='coolwarm', alpha=0.3)
axes[1].scatter(X_vis_right[:, 0], X_vis_right[:, 1], c=y_vis_right, cmap='coolwarm', edgecolors='k')
axes[1].set_xlim(x_min, x_max)
axes[1].set_ylim(y_min, y_max)
fig.suptitle("SVM Decision Function rbf kernel", fontsize=14)
plt.tight_layout()
plt.show()

### SVM Plotting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ---------------------------------------------
# Helper: map Malfunction → color
# ---------------------------------------------
def mal_to_color(m):
    if m == 0 or m == '0':
        return 'blue'        # healthy
    if m in ['C', 'E', 'G']:
        return 'green'       # 2.5 mm efflux
    if m in ['D', 'F']:
        return 'purple'      # 1 mm efflux
    if m in ['A', 'B']:
        return 'cyan'       # auxiliary leakage
    return 'gray'            # unknown / fallback

# ---------------------------------------------
# 1. Malfunction labels and colors
#    (no df.loc, just use train/test targets)
# ---------------------------------------------
mal_train_clean_arr = np.asarray(mal_train_clean)
mal_test_arr        = np.asarray(mal_test)

train_colors = [mal_to_color(m) for m in mal_train_clean_arr]
test_colors  = [mal_to_color(m) for m in mal_test_arr]

# ---------------------------------------------
# 2. Visible features for plotting (2D)
# ---------------------------------------------
X_vis_left  = np.asarray(X_train_clean)   # 2D features used for SVM train
y_vis_left  = np.asarray(y_train_clean)   # binary label 0/1

X_vis_right = np.asarray(X_test_scaled)   # 2D features used for SVM test
y_vis_right = np.asarray(y_test)          # binary label 0/1

# ---------------------------------------------
# 3. SVM decision surface
# ---------------------------------------------
svm_model = trained_models["SVM (SMOTE)"]["model"]

x_min = min(X_vis_left[:, 0].min(), X_vis_right[:, 0].min()) - 1
x_max = max(X_vis_left[:, 0].max(), X_vis_right[:, 0].max()) + 1
y_min = min(X_vis_left[:, 1].min(), X_vis_right[:, 1].min()) - 1
y_max = max(X_vis_left[:, 1].max(), X_vis_right[:, 1].max()) + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid_points = np.c_[xx.ravel(), yy.ravel()]
Z_decision = svm_model.decision_function(grid_points).reshape(xx.shape)
Z_predict  = svm_model.predict(grid_points).reshape(xx.shape)

# ---------------------------------------------
# 4. Plot side by side
# ---------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ===== LEFT: Training / RNN-clean =====
axes[0].contourf(xx, yy, Z_decision, levels=50, cmap='coolwarm', alpha=0.7)
axes[0].contour(xx, yy, Z_decision, levels=[0], linewidths=2, colors='black')

axes[0].scatter(
    X_vis_left[:, 0],
    X_vis_left[:, 1],
    c=train_colors,
    edgecolors='k'
)

# annotate only leakage points (label == 1)
for x_coord, y_coord, label, m in zip(
    X_vis_left[:, 0],
    X_vis_left[:, 1],
    y_vis_left,
    mal_train_clean_arr
):
    if label == 1:
        axes[0].annotate(
            str(m),
            (x_coord, y_coord),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=8,
            color="black"
        )

axes[0].set_title("Training Data", fontsize=14)
axes[0].set_xlim(x_min, x_max)
axes[0].set_ylim(y_min, y_max)

# ===== RIGHT: Test =====
axes[1].contourf(xx, yy, Z_predict, cmap='coolwarm', alpha=0.3)

axes[1].scatter(
    X_vis_right[:, 0],
    X_vis_right[:, 1],
    c=test_colors,
    edgecolors='k'
)

for x_coord, y_coord, label, m in zip(
    X_vis_right[:, 0],
    X_vis_right[:, 1],
    y_vis_right,
    mal_test_arr
):
    if label == 1:
        axes[1].annotate(
            str(m),
            (x_coord, y_coord),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=8,
            color="black"
        )

axes[1].set_title("Test Data",fontsize=14)
axes[1].set_xlim(x_min, x_max)
axes[1].set_ylim(y_min, y_max)

# Legend for malfunction categories
legend_elements = [
    Patch(facecolor='blue',   edgecolor='k', label='0 (healthy)'),
    Patch(facecolor='green',  edgecolor='k', label='C,E,G (Efflux 2.5 mm)'),
    Patch(facecolor='purple', edgecolor='k', label='D,F (Efflux 1 mm)'),
    Patch(facecolor='cyan',  edgecolor='k', label='A,B (Auxiliary leakage)'),
]
axes[1].legend(handles=legend_elements, loc='upper right', fontsize=12)

fig.suptitle("SVM Decision Function with Malfunction Label", fontsize=18)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ---------------------------------------------
# Helper: map Malfunction → color
# ---------------------------------------------
def mal_to_color(m):
    if m == 0 or m == '0':
        return 'blue'        # healthy
    if m in ['C', 'E', 'G']:
        return 'green'       # 2.5 mm efflux
    if m in ['D', 'F']:
        return 'purple'      # 1 mm efflux
    if m in ['A', 'B']:
        return 'cyan'       # auxiliary leakage
    return 'gray'            # unknown / fallback

# ---------------------------------------------
# 1. Malfunction labels and colors
#    (no df.loc, just use train/test targets)
# ---------------------------------------------
mal_train_clean_arr = np.asarray(mal_train_clean)
mal_test_arr        = np.asarray(mal_test)

train_colors = [mal_to_color(m) for m in mal_train_clean_arr]
test_colors  = [mal_to_color(m) for m in mal_test_arr]

# ---------------------------------------------
# 2. Visible features for plotting (2D)
# ---------------------------------------------
X_vis_left  = np.asarray(X_train_clean)   # 2D features used for SVM train
y_vis_left  = np.asarray(y_train_clean)   # binary label 0/1

X_vis_right = np.asarray(X_test_scaled)   # 2D features used for SVM test
y_vis_right = np.asarray(y_test)          # binary label 0/1

# ---------------------------------------------
# 3. SVM decision surface
# ---------------------------------------------
svm_model = trained_models["SVM_rbf (SMOTE)"]["model"]

x_min = min(X_vis_left[:, 0].min(), X_vis_right[:, 0].min()) - 1
x_max = max(X_vis_left[:, 0].max(), X_vis_right[:, 0].max()) + 1
y_min = min(X_vis_left[:, 1].min(), X_vis_right[:, 1].min()) - 1
y_max = max(X_vis_left[:, 1].max(), X_vis_right[:, 1].max()) + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid_points = np.c_[xx.ravel(), yy.ravel()]
Z_decision = svm_model.decision_function(grid_points).reshape(xx.shape)
Z_predict  = svm_model.predict(grid_points).reshape(xx.shape)

# ---------------------------------------------
# 4. Plot side by side
# ---------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ===== LEFT: Training / RNN-clean =====
axes[0].contourf(xx, yy, Z_decision, levels=50, cmap='coolwarm', alpha=0.7)
axes[0].contour(xx, yy, Z_decision, levels=[0], linewidths=2, colors='black')

axes[0].scatter(
    X_vis_left[:, 0],
    X_vis_left[:, 1],
    c=train_colors,
    edgecolors='k'
)

# annotate only leakage points (label == 1)
for x_coord, y_coord, label, m in zip(
    X_vis_left[:, 0],
    X_vis_left[:, 1],
    y_vis_left,
    mal_train_clean_arr
):
    if label == 1:
        axes[0].annotate(
            str(m),
            (x_coord, y_coord),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=8,
            color="black"
        )

axes[0].set_title("Training Data", fontsize=14)
axes[0].set_xlim(x_min, x_max)
axes[0].set_ylim(y_min, y_max)

# ===== RIGHT: Test =====
axes[1].contourf(xx, yy, Z_predict, cmap='coolwarm', alpha=0.3)

axes[1].scatter(
    X_vis_right[:, 0],
    X_vis_right[:, 1],
    c=test_colors,
    edgecolors='k'
)

for x_coord, y_coord, label, m in zip(
    X_vis_right[:, 0],
    X_vis_right[:, 1],
    y_vis_right,
    mal_test_arr
):
    if label == 1:
        axes[1].annotate(
            str(m),
            (x_coord, y_coord),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=8,
            color="black"
        )

axes[1].set_title("Test Data",fontsize=14)
axes[1].set_xlim(x_min, x_max)
axes[1].set_ylim(y_min, y_max)

# Legend for malfunction categories
legend_elements = [
    Patch(facecolor='blue',   edgecolor='k', label='0 (healthy)'),
    Patch(facecolor='green',  edgecolor='k', label='C,E,G (Efflux 2.5 mm)'),
    Patch(facecolor='purple', edgecolor='k', label='D,F (Efflux 1 mm)'),
    Patch(facecolor='cyan',  edgecolor='k', label='A,B (Auxiliary leakage)'),
]
axes[1].legend(handles=legend_elements, loc='upper right', fontsize=12)

fig.suptitle("SVM Decision Function with Malfunction Label", fontsize=18)
plt.tight_layout()
plt.show()
